In [1]:
import os
print(os.getcwd())          # shows current folder
print(os.listdir())         # lists files Python can see here

import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.chart import BarChart, Reference


def generate_ewards_report(input_csv, output_xlsx, date_label):
    df = pd.read_csv(input_csv)

    PURPLE = "FF7030A0"
    LIGHT_PURPLE = "FFE5D6F0"
    WHITE_BOLD = Font(bold=True, color="FFFFFFFF", size=11)
    TITLE_FONT = Font(bold=True, color="FFFFFFFF", size=16)
    SUBTITLE_FONT = Font(bold=False, color="FFFFFFFF", size=10)

    def fill(c):
        return PatternFill("solid", fgColor=c)

    # ---- aggregate points redeemed per restaurant ----
    # FIX: points repeat on every item row of an invoice, so keep only the
    # FIRST row of each invoice before summing (once per invoice)
    inv = df.drop_duplicates(subset=["Restaurant", "Date", "Invoice"], keep="first")

    summary = inv.groupby("Restaurant", as_index=False)["Ewards Point Redeemed"].sum()
    summary = summary.rename(columns={"Ewards Point Redeemed": "Ewards Points Redeemed"})
    summary = summary.sort_values("Ewards Points Redeemed", ascending=False, kind="mergesort").reset_index(drop=True)
    summary.insert(0, "Rank", range(1, len(summary) + 1))

    wb = Workbook()

    # ================= SHEET 1: EWARDS POINTS REDEEMED =================
    ws1 = wb.active
    ws1.title = "Ewards Points Redeemed"

    ws1.merge_cells("A1:C1")
    c = ws1["A1"]
    c.value = "EWARDS POINTS REDEEMED — OUTLET-WISE"
    c.font = TITLE_FONT; c.fill = fill(PURPLE)
    c.alignment = Alignment(horizontal="left", vertical="center")

    ws1.merge_cells("A2:C2")
    c = ws1["A2"]
    c.value = f"Report Date: {date_label}  |  Sorted by Points Redeemed (High to Low)  |  Points counted once per invoice"
    c.font = SUBTITLE_FONT; c.fill = fill(PURPLE)

    HEADER_ROW = 4
    headers = ["Rank", "Restaurant", "Ewards Points Redeemed"]
    for i, h in enumerate(headers, start=1):
        cell = ws1.cell(row=HEADER_ROW, column=i, value=h)
        cell.font = WHITE_BOLD; cell.fill = fill(PURPLE)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    for r_idx, row in enumerate(summary.itertuples(index=False), start=HEADER_ROW + 1):
        for c_idx, val in enumerate(row, start=1):
            cell = ws1.cell(row=r_idx, column=c_idx, value=val)
            if c_idx == 3:
                cell.number_format = "#,##0"

    total_row = HEADER_ROW + 1 + len(summary)
    ws1.cell(row=total_row, column=2, value="GRAND TOTAL")
    f = ws1.cell(row=total_row, column=3, value=f"=SUM(C{HEADER_ROW+1}:C{total_row-1})")
    f.number_format = "#,##0"
    for c_idx in range(1, 4):
        ws1.cell(row=total_row, column=c_idx).fill = fill(LIGHT_PURPLE)

    ws1.conditional_formatting.add(f"C{HEADER_ROW+1}:C{total_row-1}", ColorScaleRule(
        start_type="min", start_color="FFFFFFFF", end_type="max", end_color=PURPLE))

    chart = BarChart()
    chart.type = "col"
    chart.title = "Top 10 Outlets — Ewards Points Redeemed"
    chart.style = 10
    data_ref = Reference(ws1, min_col=3, min_row=HEADER_ROW, max_row=HEADER_ROW + 10)
    cats_ref = Reference(ws1, min_col=2, min_row=HEADER_ROW + 1, max_row=HEADER_ROW + 10)
    chart.add_data(data_ref, titles_from_data=True)
    chart.set_categories(cats_ref)
    chart.width, chart.height = 20, 10
    ws1.add_chart(chart, "D5")

    ws1.freeze_panes = "A5"
    ws1.column_dimensions["A"].width = 8
    ws1.column_dimensions["B"].width = 38
    ws1.column_dimensions["C"].width = 26

    # ================= SHEET 2: RAW DATA =================
    ws2 = wb.create_sheet("Raw Data")
    raw_cols = list(df.columns)
    for i, h in enumerate(raw_cols, start=1):
        cell = ws2.cell(row=1, column=i, value=h)
        cell.font = WHITE_BOLD; cell.fill = fill(PURPLE)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for r_idx, row in enumerate(df.itertuples(index=False), start=2):
        for c_idx, val in enumerate(row, start=1):
            ws2.cell(row=r_idx, column=c_idx, value=val)
    ws2.freeze_panes = "A2"
    ws2.column_dimensions["A"].width = 32

    wb.save(output_xlsx)
    print("Saved:", output_xlsx)

/home/945437b4-bbfb-4902-9dbf-ae0fc2eae44c/eward points 
['Ewards_Points_Redeemed_24November2026.xlsx', 'monthly report.ipynb', 'combine code.ipynb', 'Ewards_Points_Redeemed_25November2026.xlsx', '.ipynb_checkpoints', 'apsara_sales_reports_Customer Details Report_Table (1).csv', 'weekly report.ipynb', 'apsara_sales_reports_Customer Details Report_Table (3).csv', 'apsara_sales_reports_Customer Details Report_Table (2).csv']


In [2]:
generate_ewards_report(
    input_csv="apsara_sales_reports_Customer Details Report_Table (3).csv",
    output_xlsx="Ewards_Points_Redeemed_26November2026.xlsx",
    date_label="2026-11-26"
)

Saved: Ewards_Points_Redeemed_26November2026.xlsx
